In [1]:
import json
import yaml
from pathlib import Path
from statistics import mean
import pandas as pd

def load_yaml(path: str | Path) -> dict:
    """Memuat file YAML; raise jika tidak ada."""
    filepath = Path(path)
    if not filepath.is_file():
        raise FileNotFoundError(f"File konfigurasi tidak ditemukan: {filepath}")
    with open(filepath, "r", encoding="utf-8") as f:
        return yaml.safe_load(f) or {}

def load_json(path: str | Path):
    """Memuat file JSON."""
    filepath = Path(path)
    if not filepath.is_file():
        raise FileNotFoundError(f"File JSON tidak ditemukan: {filepath}")
    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)

def _safe_mean(values):
    """Rata-rata yang aman terhadap list kosong / nilai None."""
    vals = [v for v in values if isinstance(v, (int, float))]
    return mean(vals) if vals else float("nan")

def get_rouge_score(method_num: str, path: str) -> dict:
    """Memuat skor ROUGE (sudah berupa rata-rata/agregat)."""
    new_path = Path(f"{method_num}_outputs") / path / "rouge.json"
    return load_json(new_path)

def reformat_unieval(unieval: list) -> dict:
    """Mengembalikan RATA-RATA tiap metrik UniEval (bukan list per-instance)."""
    if not unieval:
        return {"coherence": float("nan"), "relevance": float("nan"),
                "fluency": float("nan"), "consistency": float("nan")}
    return {
        "coherence":   _safe_mean([item.get("coherence")   for item in unieval]),
        "relevance":   _safe_mean([item.get("relevance")   for item in unieval]),
        "fluency":     _safe_mean([item.get("fluency")     for item in unieval]),
        "consistency": _safe_mean([item.get("consistency") for item in unieval]),
    }

def get_unieval_score(method_num: str, path: str) -> dict:
    """Memuat skor UniEval lalu merata-ratakan."""
    new_path = Path(f"{method_num}_outputs") / path / "unieval.json"
    return reformat_unieval(load_json(new_path))

def get_align_score(method_num: str, path: str) -> dict:
    """Memuat AlignScore; kembalikan RATA-RATA (file bisa berupa list per-instance atau skalar)."""
    new_path = Path(f"{method_num}_outputs") / path / "alignscore.json"
    data = load_json(new_path)
    if isinstance(data, list):
        return {"align_score": _safe_mean(data)}
    if isinstance(data, dict):
        # ambil nilai numerik pertama, atau rata-rata jika berisi list
        for v in data.values():
            if isinstance(v, list):
                return {"align_score": _safe_mean(v)}
            if isinstance(v, (int, float)):
                return {"align_score": float(v)}
        return {"align_score": float("nan")}
    return {"align_score": float(data)}

def read_result(method_num: str) -> pd.DataFrame:
    """Membaca semua eksperimen sebuah method -> DataFrame rata-rata per eksperimen."""
    method_cfg = load_yaml(f"configs/methods/{method_num}.yaml")
    dataset_path = method_cfg.get("datasets")
    if dataset_path:
        load_yaml(f"configs/datasets/{dataset_path}")  # divalidasi keberadaannya

    folder_list = []
    for exp in method_cfg.get("experiments", []):
        exp_code = exp.get("code")
        if exp_code:
            exp_cfg = load_yaml(f"configs/experiments/{exp_code}")
            folder_list.append(exp_cfg.get("output_name"))

    rows = []
    for folder in folder_list:
        rows.append({
            "nama_file": folder,
            **get_rouge_score(method_num, folder),
            **get_unieval_score(method_num, folder),
            **get_align_score(method_num, folder),
        })
    return pd.DataFrame(rows)


In [2]:
# main() dipertahankan sebagai alias read_result agar sel-sel di bawah tetap jalan.
def main(method_num: str) -> pd.DataFrame:
    return read_result(method_num)


In [3]:
#600 data / 5 epoch 
df1 = main('method1')
df1['nama_file'] = df1['nama_file'] + '_400'
df1

,nama_file,rouge1,rouge2,rougeL,rougeLsum,bertscore_precision,bertscore_recall,bertscore_f1,coherence,relevance,fluency,consistency,align_score
0,bart_warmup_400,0.581162,0.305920,0.398279,0.553361,0.921032,0.903225,0.911961,0.821406,0.766033,0.802422,0.854921,0.673493
1,bart_finetune_400,0.592704,0.314276,0.409494,0.565731,0.921844,0.905697,0.913645,0.815821,0.754930,0.786609,0.848244,0.691155
2,bart_margin_part_400,0.570916,0.294534,0.389045,0.543626,0.919492,0.900542,0.909850,0.857363,0.787727,0.808341,0.859553,0.710620
3,bart_margin_full_400,0.576675,0.299329,0.388801,0.549671,0.919571,0.903249,0.911267,0.844120,0.799304,0.792826,0.854640,0.701936


In [4]:
# 2000 data 5 epoch
df2 = main('method2')
df2['nama_file'] = df2['nama_file'] + '_600'
df2

,nama_file,rouge1,rouge2,rougeL,rougeLsum,bertscore_precision,bertscore_recall,bertscore_f1,coherence,relevance,fluency,consistency,align_score
0,bart_warmup_600,0.604693,0.330307,0.428535,0.576452,0.923593,0.908883,0.916113,0.842863,0.793188,0.802736,0.863700,0.719119
1,bart_finetune_600,0.617088,0.341240,0.440521,0.587923,0.924185,0.910588,0.917268,0.881263,0.847122,0.827528,0.885667,0.756615
2,bart_margin_part_600,0.602813,0.329918,0.428004,0.574378,0.923738,0.906481,0.914947,0.868740,0.845499,0.832692,0.883993,0.746555
3,bart_margin_full_600,0.603766,0.330054,0.430043,0.574479,0.921854,0.906985,0.914281,0.858033,0.840385,0.826611,0.881467,0.745995


In [5]:
# 2000 data / 10 epoch

df3 = main('method3')
df3['nama_file'] = df3['nama_file'] + '_400'
df3

,nama_file,rouge1,rouge2,rougeL,rougeLsum,bertscore_precision,bertscore_recall,bertscore_f1,coherence,relevance,fluency,consistency,align_score
0,bart_warmup_400,0.616873,0.342999,0.444104,0.588786,0.924609,0.910543,0.917449,0.859771,0.826522,0.822538,0.874474,0.750973
1,bart_finetune_400,0.625396,0.353582,0.459976,0.595883,0.923189,0.912447,0.917718,0.878989,0.848142,0.826910,0.884424,0.756201
2,bart_margin_part_400,0.614073,0.339015,0.446373,0.584249,0.923942,0.909367,0.916519,0.910145,0.889828,0.847718,0.896615,0.768769
3,bart_margin_full_400,0.623736,0.345504,0.454172,0.593882,0.923172,0.912569,0.917777,0.901588,0.882837,0.848000,0.891046,0.772386


In [6]:
# 2000 data / 10 epoch / max input : 350

df4 = main('method4')
df4['nama_file'] = df4['nama_file'] + '_400'
df4

,nama_file,rouge1,rouge2,rougeL,rougeLsum,bertscore_precision,bertscore_recall,bertscore_f1,coherence,relevance,fluency,consistency,align_score
0,bart_warmup_400,0.621260,0.346358,0.448949,0.592207,0.925151,0.912039,0.918482,0.876964,0.849179,0.823657,0.887898,0.777627
1,bart_finetune_400,0.626976,0.355174,0.462766,0.596924,0.924766,0.912827,0.918683,0.890049,0.860057,0.826762,0.897782,0.787102
2,bart_margin_part_400,0.617531,0.346704,0.455655,0.589063,0.924975,0.909705,0.917189,0.906031,0.898035,0.842269,0.899592,0.794544
3,bart_margin_full_400,0.626461,0.353618,0.461165,0.596242,0.924023,0.912665,0.918249,0.898036,0.881901,0.839650,0.895702,0.778933


In [7]:
df6 = main('method6')
df6['nama_file'] = df6['nama_file'] + '_trial'
df6

,nama_file,rouge1,rouge2,rougeL,rougeLsum,bertscore_precision,bertscore_recall,bertscore_f1,coherence,relevance,fluency,consistency,align_score
0,qwen_5_warmup_trial,0.537636,0.243237,0.384386,0.505244,0.892285,0.912205,0.902005,0.907387,0.892664,0.866361,0.809857,0.687695
1,qwen_5_finetune_trial,0.536210,0.248604,0.387788,0.506163,0.893687,0.913258,0.903245,0.922703,0.884769,0.864991,0.794219,0.687962
2,qwen_5_margin_part_trial,0.539097,0.258744,0.395494,0.511472,0.896762,0.914786,0.905612,0.911466,0.879847,0.875618,0.819406,0.688249
3,qwen_5_margin_full_trial,0.390087,0.173939,0.266852,0.357066,0.861328,0.851741,0.856033,0.689765,0.634470,0.569958,0.753715,0.712353


In [8]:
df7 = main('method7')
df7['nama_file'] = df7['nama_file'] + '_1200'
df7

,nama_file,rouge1,rouge2,rougeL,rougeLsum,bertscore_precision,bertscore_recall,bertscore_f1,coherence,relevance,fluency,consistency,align_score
0,qwen_5_warmup_1200,0.527107,0.250092,0.379071,0.495456,0.893842,0.904986,0.899144,0.906686,0.888350,0.863191,0.811908,0.708870
1,qwen_5_finetune_1200,0.528744,0.254342,0.382711,0.499890,0.896609,0.906723,0.901434,0.902179,0.875252,0.868586,0.812995,0.719162
2,qwen_5_margin_part_1200,0.532084,0.259009,0.387689,0.501396,0.898897,0.906449,0.902475,0.906427,0.881739,0.872569,0.816017,0.717504
3,qwen_5_margin_full_1200,0.401182,0.177302,0.267555,0.365647,0.859382,0.855131,0.856633,0.687333,0.620887,0.619254,0.747025,0.729762


In [9]:
df8 = main('method8')
df8['nama_file'] = df8['nama_file'] + '_8'
df8

,nama_file,rouge1,rouge2,rougeL,rougeLsum,bertscore_precision,bertscore_recall,bertscore_f1,coherence,relevance,fluency,consistency,align_score
0,qwen_5_warmup_8,0.547166,0.280279,0.409484,0.517850,0.896294,0.905305,0.900473,0.864850,0.832351,0.821066,0.796861,0.687581
1,qwen_5_finetune_8,0.541755,0.279762,0.404369,0.511835,0.901051,0.902794,0.901632,0.873816,0.844606,0.834225,0.806038,0.686157
2,qwen_5_margin_part_8,0.556787,0.287307,0.416830,0.526742,0.903345,0.907584,0.905191,0.886471,0.866418,0.837664,0.816335,0.696550
3,qwen_5_margin_full_8,0.414460,0.192923,0.280137,0.379827,0.864862,0.859176,0.861410,0.703934,0.633183,0.622020,0.762430,0.737971


In [10]:
df9 = main('method9')
df9['nama_file'] = df9['nama_file'] + '_9'
df9

FileNotFoundError: File JSON tidak ditemukan: method9_outputs/qwen_5_warmup/rouge.json

In [ ]:
rouge_test = get_rouge_score('method9', 'bart_margin_part/test')
rouge_test

In [ ]:
unieval = get_unieval_score('method9', 'bart_margin_part/test')
unieval

In [ ]:
align_score = get_align_score('method9', 'bart_margin_part/test')
align_score

In [ ]:
df_concat = pd.concat([df1, df2, df3, df6, df7, df8])
df_concat

In [ ]:
def sort_values_by_metric(df, metric):
    return df.sort_values(by=metric, ascending=False)['nama_file']

In [ ]:
print(sort_values_by_metric(df_concat, 'align_score'))

In [ ]:
print(sort_values_by_metric(df_concat, 'consistency'))

In [ ]:
print(sort_values_by_metric(df_concat, 'relevance'))

In [ ]:
print(sort_values_by_metric(df_concat, 'coherence'))